# Fase 4.4 — Análisis de sensibilidad preespecificados e inferencia de rechazados

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Las conclusiones del notebook 10 dependen de decisiones que podrían haberse tomado de otra forma: la definición del evento, el tratamiento de los indeterminados, la reclasificación de 131 y 250, la regla de refinanciación, la cohorte de arranque, las variables. Aquí repito la validación cambiando **una decisión a la vez** y miro si las conclusiones resisten.

Todos los escenarios están listados en la sección 7 de la enmienda v1.1. Siguiendo el protocolo, cada sensibilidad usa:

- una validación cruzada agrupada de 5 pliegues × 10 repeticiones, con particiones propias de cada población;
- el modelo elegido (M2), M3 y la tasa base M0a;
- IC por *bootstrap* de clientes con 1.000 réplicas.

Al final van las curvas de τ (M3) y del peso de Home Credit (M4), la estabilidad de la selección de variables y la inferencia de rechazados.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from tesis_v2 import config, contexto, diseno, figuras, io_datos, metricas as mt, modelos, objetivo, rechazados

warnings.filterwarnings("ignore")
pd.set_option("display.width", 190, "display.max_columns", 30, "display.max_colwidth", 60)
figuras.estilo()

D0, ctx0, ds0 = contexto.cargar()
REP, BOOT = 10, 1000


def evaluar(D, ctx, creadores, n_rep=REP, B=BOOT, semilla=config.SEMILLA):
    F = diseno.particiones(D["grupo_cliente"], D["y"], n_rep=n_rep, semilla=semilla)
    Ps = {k: modelos.validar(D, ctx, c, F, n_jobs=10)[0] for k, c in creadores.items()}
    boot = mt.bootstrap(Ps, D["y"].to_numpy(), D["grupo_cliente"].to_numpy(), B=B, semilla=semilla)
    tab = mt.tabla_resultados(Ps, D["y"].to_numpy(), boot)
    return Ps, boot, tab


def resumen(nombre, D, ctx, creadores=None, extra_m2=None):
    creadores = creadores or {"M0a": modelos.M0aTasaBase, "M2": lambda: modelos.M2Penalizada(extra=extra_m2),
                              "M3": modelos.M3PriorHomeCredit}
    Ps, boot, tab = evaluar(D, ctx, creadores)
    T = tab.set_index("modelo")
    fila = {"escenario": nombre, "creditos": len(D), "eventos": int(D["y"].sum()), "tasa_%": round(100 * D["y"].mean(), 2)}
    for k in ["M2", "M3"]:
        c = mt.comparacion_pareada(boot, tab, k, "M0a", "log_loss")
        fila[f"auc_{k}"] = round(T.loc[k, "auc"], 3)
        fila[f"auc_{k}_ic"] = T.loc[k, "auc_ic"]
        fila[f"dlogloss_{k}_vs_M0a_x1000"] = round(1000 * c["delta"], 2)
        fila[f"dlogloss_{k}_ic_x1000"] = f"{1000 * c['ic_inf']:+.1f} a {1000 * c['ic_sup']:+.1f}"
        fila[f"pendiente_{k}"] = round(T.loc[k, "pendiente_cal"], 2)
    c = mt.comparacion_pareada(boot, tab, "M3", "M2", "log_loss")
    fila["dlogloss_M3_vs_M2_x1000"] = f"{1000 * c['delta']:+.2f} [{1000 * c['ic_inf']:+.1f}; {1000 * c['ic_sup']:+.1f}]"
    coef = modelos.M2Penalizada(extra=extra_m2).ajustar(D, ctx).coeficientes(D)
    for v in ["log_emp_previos", "log_monto", "edad_t0", "meses_cliente"]:
        fila[f"coef_{v}"] = round(coef[v], 3)
    if extra_m2:
        for v in extra_m2:
            fila[f"coef_{v}"] = round(coef[v], 3)
    return fila

## 1. Sensibilidad de las conclusiones a cada decisión de diseño

Para la regla D4 (refinanciación no enlazada) reconstruyo la etiqueta con las otras dos ventanas del protocolo y sustituyo las columnas de desenlace en el dataset. Los predictores no cambian, porque solo dependen de información anterior a t0.

In [2]:
def dataset_con_ventana(ventana):
    t = objetivo.cargar_y_reconstruir(ventana_refi=ventana)["objetivo"].set_index("cid")
    ds = ds0.set_index("cid").copy()
    cols = ["cierre_v2", "madura_6m", "madura_12m", "y_90_12m", "y_60_12m", "y_30_6m", "y_90_12m_B",
            "evento_90", "tiempo_90"]
    ds[cols] = t.reindex(ds.index)[cols]
    return ds.reset_index()


ESCENARIOS = [
    ("Principal (v1.1: 733/23)", dict()),
    ("Etiqueta original de la Fase 1 (731/21)", dict(etiqueta="original")),
    ("Evento 60+/12m", dict(objetivo="y_60_12m")),
    ("Evento 30+/6m", dict(objetivo="y_30_6m")),
    ("Evento 90+/12m, variante B", dict(objetivo="y_90_12m_B")),
    ("Indeterminados = no evento", dict(indeterminados="no_evento")),
    ("Indeterminados = evento", dict(indeterminados="evento")),
    ("Sin la cohorte 2022", dict(excluir_cohortes=(2021, 2022))),
    ("Regla D4 con ventana −7/+30", dict(ds=dataset_con_ventana((-7, 30)))),
    ("Regla D4 con ventana −30/+60", dict(ds=dataset_con_ventana((-30, 60)))),
    ("Casos completos (sin imputar edad)", dict(filtro=lambda d: d["edad_t0"].notna())),
]
filas = []
for nombre, kw in ESCENARIOS:
    D, ctx, _ = contexto.cargar(**kw)
    filas.append(resumen(nombre, D, ctx))
    print(f"{nombre:42s} n={len(D)} eventos={int(D['y'].sum())}")
D, ctx, _ = contexto.cargar()
filas.append(resumen("Con recaudo directo como variable adicional", D, ctx,
                     extra_m2={"recaudo_directo": lambda d: (d["recaudo"] != "empresarial").astype(float)}))
sens = pd.DataFrame(filas)
io_datos.guardar_resultado(sens, "fase4", "sensibilidades")
sens[["escenario", "creditos", "eventos", "auc_M2", "auc_M2_ic", "dlogloss_M2_vs_M0a_x1000", "dlogloss_M2_ic_x1000",
      "pendiente_M2", "auc_M3", "dlogloss_M3_vs_M2_x1000"]]

Principal (v1.1: 733/23)                   n=733 eventos=23


Etiqueta original de la Fase 1 (731/21)    n=731 eventos=21


Evento 60+/12m                             n=733 eventos=29


Evento 30+/6m                              n=790 eventos=64


Evento 90+/12m, variante B                 n=733 eventos=126


Indeterminados = no evento                 n=773 eventos=23


... (2 outputs left out)

Regla D4 con ventana −7/+30                n=723 eventos=23


Regla D4 con ventana −30/+60               n=744 eventos=23


Casos completos (sin imputar edad)         n=730 eventos=23


,escenario,creditos,eventos,auc_M2,auc_M2_ic,dlogloss_M2_vs_M0a_x1000,dlogloss_M2_ic_x1000,pendiente_M2,auc_M3,dlogloss_M3_vs_M2_x1000
0,Principal (v1.1: 733/23),733,23,0.628,0.505–0.749,-0.55,-8.0 a +7.8,0.56,0.650,-1.94 [-5.1; +0.7]
1,Etiqueta original de la Fase 1 (731/21),731,21,0.617,0.478–0.746,-0.14,-6.5 a +7.5,0.61,0.642,-0.93 [-4.2; +2.3]
2,Evento 60+/12m,733,29,0.633,0.538–0.734,-2.64,-10.2 a +4.9,0.76,0.644,-1.25 [-4.5; +1.7]
3,Evento 30+/6m,790,64,0.542,0.471–0.610,0.97,-3.5 a +5.9,0.43,0.547,+1.15 [-2.2; +4.3]
4,"Evento 90+/12m, variante B",733,126,0.615,0.564–0.664,-14.57,-29.8 a +0.3,0.82,0.613,+0.19 [-2.4; +2.8]
5,Indeterminados = no evento,773,23,0.620,0.493–0.733,-0.53,-6.4 a +6.7,0.61,0.641,-2.01 [-4.8; +0.5]
6,Indeterminados = evento,773,63,0.623,0.543–0.699,-4.52,-14.3 a +5.3,0.85,0.633,-1.38 [-5.4; +2.8]
7,Sin la cohorte 2022,684,17,0.498,0.392–0.593,4.50,-0.7 a +13.4,-0.69,0.552,-0.79 [-4.7; +2.6]
8,Regla D4 con ventana −7/+30,723,23,0.624,0.502–0.747,-0.01,-8.4 a +9.6,0.51,0.650,-2.57 [-5.9; +0.1]
9,Regla D4 con ventana −30/+60,744,23,0.629,0.501–0.751,-0.69,-8.7 a +7.9,0.60,0.657,-2.34 [-5.1; +0.1]


In [3]:
sens[["escenario"] + [c for c in sens.columns if c.startswith("coef_")]]

,escenario,coef_log_emp_previos,coef_log_monto,coef_edad_t0,coef_meses_cliente,coef_recaudo_directo
0,Principal (v1.1: 733/23),-0.329,0.239,-0.199,-0.168,NaN
1,Etiqueta original de la Fase 1 (731/21),-0.287,0.171,-0.099,-0.090,NaN
2,Evento 60+/12m,-0.351,0.083,-0.039,-0.167,NaN
3,Evento 30+/6m,-0.195,0.000,0.000,0.000,NaN
4,"Evento 90+/12m, variante B",-0.031,-0.066,0.000,-1.151,NaN
5,Indeterminados = no evento,-0.174,0.122,-0.090,-0.076,NaN
6,Indeterminados = evento,-0.094,0.042,-0.034,-0.045,NaN
7,Sin la cohorte 2022,0.000,0.000,0.000,0.000,NaN
8,Regla D4 con ventana −7/+30,-0.187,0.123,-0.090,-0.078,NaN
9,Regla D4 con ventana −30/+60,-0.061,0.038,-0.027,-0.030,NaN


**Lo que leo en la tabla de sensibilidades.**

1. **La conclusión central resiste en casi todos los escenarios.** La mejora de M2 sobre la tasa base nunca es demostrable: todos los IC del Δ log loss incluyen 0. El AUC de M2 se mueve entre 0,61 y 0,63, con IC que rozan 0,5, en las variantes de etiqueta (original 731/21, 60+, indeterminados en ambos extremos, ventanas D4, casos completos).
2. **Volver a la etiqueta original (731/21)** baja un poco el AUC (0,615) y no cambia ninguna conclusión. La reclasificación de 131 y 250 no es la que sostiene los resultados.
3. **Las definiciones más amplias del evento** (60+/12m e indeterminados como evento) dan más eventos, mejor calibración (pendiente de 0,76 a 0,85) y un Δ más negativo, aunque todavía no significativo. La variante B (126 eventos) es la que más se acerca: Δ −14,6 × 10⁻³ con IC [−29,8; +0,3]. Con más eventos, el poco poder que tenemos empieza a notarse. Esto sugiere que el problema es sobre todo de tamaño muestral.
4. **El evento 30+/6m se discrimina peor** (AUC 0,54). La mora temprana parece responder a otros mecanismos (fricción operativa del recaudo) más que al riesgo de fondo.
5. **Sin la cohorte 2022 desaparece la señal** (AUC 0,49 y Δ positivo, es decir, peor que la tasa base; el ajuste final contrae todos los coeficientes a 0). Seis de los 23 eventos están en 2022, y una parte del poder discriminante viene de las diferencias entre cohortes. Es la sensibilidad más importante del notebook: la señal del modelo es frágil y depende de la época. Es coherente con la deriva documentada en la Fase 2 y refuerza la necesidad del aprendizaje incremental.
6. **Agregar `recaudo_directo` no ayuda.** En el ajuste final su coeficiente sale exactamente 0, porque la penalización elegida lo descarta, y el AUC baja a 0,614. Excluir `recaudo` del modelo principal no costó desempeño.
7. **En casi todos los escenarios M3 queda por encima de M2 en log loss**, con IC que incluyen 0 o lo rozan (ventanas D4: límite superior +0,1). La ventaja del prior es consistente en dirección y pequeña en magnitud.
8. **Los signos de los coeficientes de M2** (empleador −, monto +, edad −, antigüedad −) se mantienen en todas las variantes que conservan la ventana de 12 meses. Lo que cambia es la magnitud, según la contracción que elija el pliegue interno. Hay dos excepciones informativas:
   - En la **variante B**, el monto cambia de signo y domina la antigüedad como cliente (−1,15). Sin controlar la exposición, el modelo aprende "cuánto tiempo lleva el crédito", no el riesgo.
   - **Sin la cohorte 2022**, el ajuste final contrae todos los coeficientes a 0 y el modelo queda reducido a la tasa base.

## 2. Curvas de la información externa: τ en M3 y peso de Home Credit en M4

Ahora miro el desempeño como función de cuánta información externa se usa:

- **M3:** fijo τ en vez de elegirlo en el pliegue interno. Los valores 0,05 y 0,1 quedan **fuera de la grilla preespecificada** y son exploratorios. Los agrego porque el pliegue interno eligió el borde inferior (τ = 0,25) en 97 de 100 ajustes, y conviene saber qué hay más allá. Como referencia incluyo M3 con centro 0, que equivale a una logística L2 con C = τ².
- **M4:** recorro la fracción del peso de entrenamiento que aporta Home Credit, desde 0 (solo FINTECH) hasta 80 % (la v1).

In [4]:
TAUS = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0]
cre = {}
for t in TAUS:
    cre[f"HC τ={t}"] = (lambda t=t: modelos.M3PriorHomeCredit(taus=[t]))
    cre[f"0 τ={t}"] = (lambda t=t: modelos.M3PriorHomeCredit(taus=[t], centro_hc=False))
FR = [0.0, 0.1, 0.25, 0.5, 0.8]
for f in FR:
    cre[f"M4 f={f}"] = (lambda f=f: modelos.M4ApiladoV1(fraccion_hc=f))
cre["M0a"] = modelos.M0aTasaBase
Pc, bootc, tabc = evaluar(D0, ctx0, cre)
io_datos.guardar_resultado(tabc.drop(columns="nombre", errors="ignore").round(5), "fase4", "curvas_tau_y_peso_hc")
ll0 = tabc.set_index("modelo").loc["M0a", "log_loss"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
T = tabc.set_index("modelo")
for centro, col, lab in [("HC", figuras.NARANJA, "Centro Home Credit"), ("0", figuras.AZUL, "Centro 0 (= L2)")]:
    ll = [T.loc[f"{centro} τ={t}", "log_loss"] - ll0 for t in TAUS]
    auc = [T.loc[f"{centro} τ={t}", "auc"] for t in TAUS]
    axes[0].plot(TAUS, 1000 * np.array(ll), "o-", color=col, label=lab)
    axes[1].plot(TAUS, auc, "o-", color=col, label=lab)
for ax in axes[:2]:
    ax.set_xscale("log")
    ax.axvspan(0.03, 0.2, color="#f2f2f2", zorder=0)
    ax.set_xlabel("Escala del prior τ (zona gris: fuera de la grilla preespecificada)")
axes[0].axhline(0, color=figuras.GRIS, lw=1)
axes[0].set_ylabel("Δ log loss frente a la tasa base (× 1.000)")
axes[1].set_ylabel("AUC")
axes[0].legend(fontsize=8)
ll4 = [T.loc[f"M4 f={f}", "log_loss"] - ll0 for f in FR]
pend4 = [T.loc[f"M4 f={f}", "pendiente_cal"] for f in FR]
axes[2].plot(100 * np.array(FR), 1000 * np.array(ll4), "o-", color=figuras.VERDE, label="Δ log loss × 1.000")
ax2 = axes[2].twinx()
ax2.plot(100 * np.array(FR), pend4, "s--", color=figuras.GRIS, label="pendiente de calibración")
ax2.set_ylabel("Pendiente de calibración")
axes[2].axhline(0, color=figuras.GRIS, lw=1)
axes[2].set_xlabel("Peso de Home Credit en el entrenamiento (%)")
axes[2].set_title("M4: datos apilados", fontsize=9)
axes[0].set_title("M3: log loss según τ", fontsize=9)
axes[1].set_title("M3: AUC según τ", fontsize=9)
fig.tight_layout()
figuras.guardar(fig, "fase4", "M07", "curvas_tau_peso_hc",
                "El log loss es mínimo en τ = 0,25 con ambos centros; por debajo, el centro Home Credit empeora porque impone direcciones que FINTECH no respalda. Apilar más Home Credit mejora M4 sin alcanzar a M3.")
tabc.set_index("modelo").loc[[c for c in cre if c != "M0a"], ["log_loss", "auc", "pendiente_cal", "intercepto_cal"]].round(4)

,log_loss,auc,pendiente_cal,intercepto_cal
modelo,,,,
HC τ=0.05,0.1418,0.5018,0.0467,-0.0004
0 τ=0.05,0.1386,0.6286,4.9860,0.0001
HC τ=0.1,0.1382,0.5834,0.8147,-0.0007
0 τ=0.1,0.1372,0.6479,2.0367,-0.0001
HC τ=0.25,0.1365,0.6508,0.7789,-0.0035
0 τ=0.25,0.1370,0.6455,0.7632,-0.0033
HC τ=0.5,0.1402,0.6379,0.4587,-0.0115
0 τ=0.5,0.1406,0.6295,0.4370,-0.0120
HC τ=1.0,0.1450,0.6180,0.2818,-0.0216


**Lectura.**

- **El τ preespecificado más pequeño (0,25) es también el óptimo.** Ir más allá del borde de la grilla no mejora. Con τ = 0,1 y 0,05, el centro Home Credit empeora con claridad (AUC 0,58 y 0,50), porque el prior impone las direcciones de Home Credit y dos de ellas contradicen lo que se ve en FINTECH (`tiene_historia` y `log_plazo`, notebook 11). El centro 0 con τ pequeño conserva el ordenamiento (AUC 0,65), pero queda sobrecontraído (pendiente de 2 a 5).
- Por eso, que el pliegue interno eligiera el borde de la grilla en 97/100 ajustes no esconde una ganancia perdida. La grilla cubría el óptimo.
- **En τ = 0,25, el centro Home Credit gana al centro 0 por muy poco** (log loss 0,1365 frente a 0,1370; AUC 0,651 frente a 0,646). El aporte de Home Credit como prior es real en dirección y marginal en magnitud.
- **En M4, más peso de Home Credit mejora el log loss y la calibración de forma monótona**, pero ni con 80 % llega al nivel de M3 con τ = 0,25 (0,1388 frente a 0,1365). Apilar es una manera menos eficiente de transferir la información.

## 3. Estabilidad de la selección de variables

Si el modelo tuviera que quedarse con pocas variables, ¿cuáles elegiría de forma estable? Ajusto una logística L1 en 500 réplicas *bootstrap* por cliente, con el C que el pliegue interno elige para L1 en los datos completos, y cuento en qué fracción de réplicas cada variable queda con coeficiente distinto de 0.

In [5]:
ml1 = modelos.M2Penalizada(penalizaciones=(1.0,)).ajustar(D0, ctx0)
C1 = ml1.elegido[0]
print(f"C de L1 elegido en el pliegue interno: {C1:.3g}")


def seleccion(idx, C):
    Db = D0.iloc[idx].reset_index(drop=True)
    return modelos._Logistica((C, 1.0)).ajustar(Db, ctx0).pipe[-1].coef_[0] != 0


idxs = list(mt.indices_bootstrap(D0["grupo_cliente"], B=500, semilla=config.SEMILLA + 4))
filas = {}
for C in sorted({round(C1, 4), 0.1}, reverse=True):
    sel = np.array(Parallel(n_jobs=10)(delayed(seleccion)(i, C) for i in idxs))
    filas[f"C={C:.3g}"] = sel.mean(axis=0)
estab = pd.DataFrame(filas, index=diseno.NUCLEO)
io_datos.guardar_resultado(estab.round(3).reset_index(names="variable"), "fase4", "estabilidad_seleccion_L1")

fig, ax = plt.subplots(figsize=(7, 4.2))
o = estab.iloc[:, -1].sort_values().index
for (c, col), dy in zip(zip(estab.columns, figuras.SECUENCIA), [-0.25, 0, 0.25]):
    ax.barh(np.arange(len(o)) + dy, estab.loc[o, c], height=0.25, color=col, label=c)
ax.set_yticks(range(len(o)), o, fontsize=8)
ax.set_xlabel("Fracción de réplicas bootstrap en que la variable queda seleccionada")
ax.set_title("Estabilidad de la selección con L1 (500 réplicas por cliente)")
ax.legend(fontsize=8)
figuras.guardar(fig, "fase4", "M08", "estabilidad_seleccion",
                "Con el C elegido, empleador, monto y edad entran en más del 80 % de las réplicas; con más penalización solo sobrevive el empleador.")
estab.round(3)

C de L1 elegido en el pliegue interno: 0.464


,C=0.464,C=0.1
log_monto,0.854,0.316
log_plazo,0.570,0.042
edad_t0,0.844,0.202
tiene_historia,0.382,0.056
log_n_previos,0.228,0.026
meses_cliente,0.544,0.072
log_peor_mora_previa,0.482,0.038
previo_adverso,0.524,0.014
previo_favorable,0.400,0.006
n_activos_t0,0.540,0.070


**Lectura.** El número de años con el empleador (`log_emp_previos`) es la única variable que sobrevive incluso con una penalización fuerte (C = 0,1 → 79 % de las réplicas). Con el C elegido entran también monto (85 %) y edad (84 %). Las variables de historia previa (n previos, peor mora, adverso/favorable) se seleccionan de forma errática, entre 23 % y 52 %. Esto es coherente con el notebook 11: el empleador es la señal más sólida, y la historia crediticia interna aporta poco una vez se controla lo demás.

## 4. Inferencia de rechazados (análisis secundario)

El modelo solo ve créditos **aprobados**. Si la política rechazó a los más riesgosos, las relaciones aprendidas pueden estar sesgadas, y la PD de la población que solicita (*through-the-door*) queda subestimada. El rechazo pasó de 5,5 % en 2023 a 27,8 % en 2025.

Construyo los predictores en t0 de las solicitudes de libranza rechazadas dentro de la misma ventana de originación, con la misma lógica de los créditos (`src/tesis_v2/rechazados.py`). Primero comparo los rechazados con los aprobados.

In [6]:
REJ = rechazados.construir(D0["t0"].min(), D0["t0"].max())
XA, XR = diseno.construir_X(D0), diseno.construir_X(REJ)
smd = pd.DataFrame({"media_aprobados": XA.mean(), "media_rechazados": XR.mean(),
                    "SMD": (XR.mean() - XA.mean()) / np.sqrt((XA.var() + XR.var()) / 2)}).round(3)
print(f"Solicitudes rechazadas en la ventana: {len(REJ)} ({REJ['t0'].dt.year.value_counts().sort_index().to_dict()})")
io_datos.guardar_resultado(smd.reset_index(names="variable"), "fase4", "rechazados_vs_aprobados")
smd

Solicitudes rechazadas en la ventana: 202 ({2023: 6, 2024: 47, 2025: 87, 2026: 62})


,media_aprobados,media_rechazados,SMD
log_monto,14.953,15.064,0.099
log_plazo,2.652,2.710,0.080
edad_t0,39.306,39.260,-0.004
tiene_historia,0.531,0.589,0.118
log_n_previos,0.541,0.613,0.124
meses_cliente,7.864,9.116,0.117
log_peor_mora_previa,0.994,1.043,0.034
previo_adverso,0.195,0.188,-0.018
previo_favorable,0.171,0.149,-0.060
n_activos_t0,0.379,0.525,0.262


**Lectura.** Los rechazados se parecen mucho a los aprobados en las variables del modelo: todas las diferencias estandarizadas (SMD) son menores que 0,3. Las mayores están en más créditos activos en t0 (+0,26) y menos tiempo con el empleador (−0,20), y ambas apuntan hacia más riesgo. La política de rechazo no parece seleccionar con base en estas variables. Más bien selecciona por capacidad de pago, listas restrictivas o documentación, que el modelo no ve, y por eso el sesgo de selección **sobre las variables observadas** debería ser moderado.

Uso **aumentación difusa** (*fuzzy augmentation*):

1. En cada pliegue ajusto M2 con los aprobados de entrenamiento y predigo la PD p̂ de los rechazados que no pertenecen a clientes del pliegue de prueba.
2. Cada rechazado entra al entrenamiento dos veces: como evento con peso min(1, k·p̂) y como no evento con el peso complementario. El factor k ∈ {1, 2, 3} expresa el supuesto de que los rechazados son igual de riesgosos, el doble o el triple de lo que el modelo de aprobados ve.
3. Reajusto M2 con la misma configuración y evalúo **solo en aprobados**, que son los únicos con desenlace.

El supuesto k no es verificable con estos datos. Por eso es una sensibilidad, no una corrección.

In [7]:
class M2Aumentado(modelos.Base):
    def __init__(self, k):
        self.k, self.nombre = k, f"M2 + rechazados (k={k})"

    def ajustar(self, D, ctx):
        base = modelos.M2Penalizada().ajustar(D, ctx)
        self.elegido = base.elegido
        R = REJ[~REJ["grupo_cliente"].isin(ctx.grupos_primarios - set(D["grupo_cliente"]))]
        q = np.minimum(1, self.k * base.predecir(R))
        Xa = pd.concat([diseno.construir_X(D), diseno.construir_X(R), diseno.construir_X(R)], ignore_index=True)
        ya = np.r_[D["y"].to_numpy(), np.ones(len(R)), np.zeros(len(R))]
        wa = np.r_[np.ones(len(D)), q, 1 - q]
        C, l1 = base.elegido
        solver = "lbfgs" if l1 == 0 else ("liblinear" if l1 == 1 else "saga")
        self.pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                  LogisticRegression(C=C, l1_ratio=l1, solver=solver, max_iter=5000))
        self.pipe.fit(Xa, ya, logisticregression__sample_weight=wa)
        return self

    def predecir(self, D):
        return self.pipe.predict_proba(diseno.construir_X(D))[:, 1]


cre_r = {"M0a": modelos.M0aTasaBase, "M2": modelos.M2Penalizada}
for k in [1, 2, 3]:
    cre_r[f"k={k}"] = (lambda k=k: M2Aumentado(k))
Pr, bootr, tabr = evaluar(D0, ctx0, cre_r)
filas = []
m2f = modelos.M2Penalizada().ajustar(D0, ctx0)
for k in ["M2", "k=1", "k=2", "k=3"]:
    m = m2f if k == "M2" else M2Aumentado(int(k[-1])).ajustar(D0, ctx0)
    coef = m.coeficientes(D0) if k == "M2" else pd.Series(m.pipe[-1].coef_[0], index=diseno.NUCLEO)
    fila = {"modelo": k, "auc_aprobados": tabr.set_index("modelo").loc[k, "auc"],
            "auc_ic": tabr.set_index("modelo").loc[k, "auc_ic"],
            "log_loss_aprobados": tabr.set_index("modelo").loc[k, "log_loss"],
            "pd_media_aprobados": m.predecir(D0).mean(), "pd_media_rechazados": m.predecir(REJ).mean()}
    n_a, n_r = len(D0), len(REJ)
    fila["pd_through_the_door"] = (n_a * fila["pd_media_aprobados"] + n_r * fila["pd_media_rechazados"]) / (n_a + n_r)
    fila.update({f"coef_{v}": coef[v] for v in diseno.NUCLEO})
    filas.append(fila)
rinf = pd.DataFrame(filas)
io_datos.guardar_resultado(rinf.round(4), "fase4", "inferencia_rechazados")

fig, ax = plt.subplots(figsize=(7.5, 4.4))
yy = np.arange(len(diseno.NUCLEO))[::-1]
for (i, fila), col, dy in zip(rinf.iterrows(), figuras.SECUENCIA, [-0.27, -0.09, 0.09, 0.27]):
    ax.scatter([fila[f"coef_{v}"] for v in diseno.NUCLEO], yy + dy, color=col, s=18, label=fila["modelo"])
ax.axvline(0, color=figuras.GRIS, lw=1)
ax.set_yticks(yy, diseno.NUCLEO, fontsize=8)
ax.set_xlabel("Coeficiente estandarizado")
ax.set_title("Coeficientes de M2 con y sin inferencia de rechazados")
ax.legend(fontsize=8)
figuras.guardar(fig, "fase4", "M09", "inferencia_rechazados",
                "La aumentación con rechazados apenas cambia la discriminación en aprobados; los coeficientes se contraen y la PD de la población que solicita sube solo bajo el supuesto k.")
rinf[["modelo", "auc_aprobados", "auc_ic", "log_loss_aprobados", "pd_media_aprobados", "pd_media_rechazados", "pd_through_the_door"]].round(4)

,modelo,auc_aprobados,auc_ic,log_loss_aprobados,pd_media_aprobados,pd_media_rechazados,pd_through_the_door
0,M2,0.6277,0.505–0.749,0.1391,0.0314,0.0329,0.0318
1,k=1,0.6255,0.502–0.747,0.1392,0.0314,0.0329,0.0317
2,k=2,0.6273,0.504–0.747,0.1400,0.0383,0.0411,0.0389
3,k=3,0.6278,0.505–0.748,0.1420,0.0450,0.0493,0.0459


**Lectura.**

- Con k = 1 la aumentación es casi neutra, como se espera: los rechazados entran con la PD del propio modelo.
- Con k = 2 y k = 3, la PD de la población que solicita sube de 3,2 % a 3,9 % y 4,6 %. La discriminación en aprobados no cambia (AUC entre 0,626 y 0,628). El log loss en aprobados empeora un poco, porque el modelo queda desplazado hacia arriba.
- Los coeficientes se mueven poco, lo cual es coherente con que los rechazados se parezcan a los aprobados.

La conclusión práctica es que el ordenamiento de M2 no depende del sesgo de selección observable. Sí queda una incertidumbre sobre el **nivel** de PD de la población que solicita, que solo podría resolverse con datos de desempeño de rechazados (por ejemplo, buró posterior) o con aprobaciones experimentales. Eso queda para la Fase 6.

In [8]:
_ = io_datos.escribir_registro("12_sensibilidades", {
    "escenarios": [e for e, _ in ESCENARIOS] + ["recaudo"], "repeticiones": REP, "bootstrap": BOOT,
    "rechazados": len(REJ), "C_L1": float(C1)})
figuras.escribir_catalogo("fase4", "Figuras de la Fase 4")

PosixPath('resultados/fase4/catalogo_figuras.md')